# Follow-Ups Exploratory Data Analysis

## Purpose

This notebook checks follow-up data. It checks missing values, IDs, dates, outcomes, and links to referrals.

## Files used

- `Follow_Ups.csv` — follow-up records
- `Referrals.csv` — referral records

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

def find_raw_data_dir(start: Path = Path.cwd()) -> Path:
    for directory in (start, *start.parents):
        for candidate in (directory / 'data' / 'raw', directory / 'data-analytics' / 'data' / 'raw'):
            if candidate.is_dir():
                return candidate
    raise FileNotFoundError('Could not locate data-analytics/data/raw')

RAW_DATA_DIR = find_raw_data_dir()

## 1. Load the data

In [ ]:
follow_ups = pd.read_csv(RAW_DATA_DIR / 'Follow_Ups.csv')
referrals = pd.read_csv(RAW_DATA_DIR / 'Referrals.csv')
pd.DataFrame({'dataset': ['Follow Ups', 'Referrals'], 'rows': [len(follow_ups), len(referrals)], 'columns': [len(follow_ups.columns), len(referrals.columns)]})

## 2. Check the file

In [ ]:
follow_ups

In [ ]:
follow_up_profile = pd.DataFrame({
    'data_type': follow_ups.dtypes.astype(str),
    'missing_count': follow_ups.isna().sum(),
    'missing_percent': follow_ups.isna().mean().mul(100).round(1),
    'unique_values': follow_ups.nunique(dropna=False),
})
follow_up_profile

In [ ]:
required_columns = ['follow_up_id', 'referral_id', 'follow_up_date', 'outcome', 'notes']
assert set(required_columns + ['next_follow_up_date']).issubset(follow_ups.columns)
assert follow_ups[required_columns].notna().all().all()
print('All required follow-up fields have values.')
print('The next follow-up date is optional when the outcome is complete.')

## 3. Check IDs and links

In [ ]:
id_checks = pd.Series({
    'duplicate_follow_up_ids': follow_ups['follow_up_id'].duplicated().sum(),
    'wrong_follow_up_id_format': (~follow_ups['follow_up_id'].str.match(r'^FUP-[0-9]{3}$', na=False)).sum(),
    'wrong_referral_id_format': (~follow_ups['referral_id'].str.match(r'^REF-[0-9]{3}$', na=False)).sum(),
    'duplicate_referral_ids': follow_ups['referral_id'].duplicated().sum(),
    'unknown_referral_ids': len(set(follow_ups['referral_id']) - set(referrals['referral_id'])),
    'exact_duplicate_rows': follow_ups.duplicated().sum(),
})
id_checks.to_frame('count')

In [ ]:
assert id_checks.eq(0).all()
print('Follow-up IDs are unique and correctly formatted.')
print('All follow-ups link to a known referral.')

## 4. Check dates

In [ ]:
follow_up_dates = follow_ups.merge(referrals[['referral_id', 'referred_at']], on='referral_id', how='left', validate='one_to_one')
for column in ['referred_at', 'follow_up_date', 'next_follow_up_date']:
    follow_up_dates[column] = pd.to_datetime(follow_up_dates[column], utc=True, errors='coerce')
date_checks = pd.Series({
    'bad_follow_up_dates': follow_up_dates['follow_up_date'].isna().sum(),
    'follow_up_before_referral': (follow_up_dates['follow_up_date'] < follow_up_dates['referred_at']).sum(),
    'next_date_before_follow_up': ((follow_up_dates['next_follow_up_date'] < follow_up_dates['follow_up_date']) & follow_up_dates['next_follow_up_date'].notna()).sum(),
    'completed_with_next_date': ((follow_up_dates['outcome'] == 'completed') & follow_up_dates['next_follow_up_date'].notna()).sum(),
    'open_without_next_date': ((follow_up_dates['outcome'] != 'completed') & follow_up_dates['next_follow_up_date'].isna()).sum(),
})
date_checks.to_frame('count')

In [ ]:
assert date_checks.eq(0).all()
print('All dates are valid and follow the right order.')
print('Completed follow-ups have no next date. Open follow-ups have a next date.')

## 5. Review outcomes

In [ ]:
outcome_counts = follow_ups['outcome'].value_counts()
outcome_counts.to_frame('follow_up_count')

In [ ]:
ax = outcome_counts.plot(kind='bar', color='#176B87', figsize=(7, 4), title='Follow-ups by outcome')
ax.set_xlabel('')
ax.set_ylabel('Follow-up count')
ax.tick_params(axis='x', rotation=0)
ax.bar_label(ax.containers[0])
plt.tight_layout()
plt.show()

## 6. Check referral coverage

In [ ]:
referral_coverage = referrals[['referral_id', 'status']].assign(has_follow_up=lambda frame: frame['referral_id'].isin(follow_ups['referral_id']))
coverage_summary = referral_coverage.groupby(['status', 'has_follow_up']).size().rename('referral_count').reset_index()
display(coverage_summary)
print('Referrals with a follow-up:', referral_coverage['has_follow_up'].sum())
print('Referrals without a follow-up:', (~referral_coverage['has_follow_up']).sum())

## 7. Findings

There are 8 follow-up rows. All required fields have values. Follow-up IDs are unique and correctly formatted. All rows link to known referrals.

Six follow-ups are complete. Two follow-ups need more contact. The 2 open follow-ups have a next follow-up date. The 6 complete follow-ups do not have a next follow-up date. This is correct.

Eight of the 12 referrals have follow-up records. Four referrals do not have follow-up records yet.

## Next steps

- Keep the ID, date, duplicate, and referral link checks.
- Follow up on the 2 contacted records on their next dates.
- Review the 4 referrals with no follow-up records.
- Add more records before looking for trends.